Importando as bibliotecas

In [363]:
import pandas as pd
import numpy as np
import uuid6
import psycopg2
import os
import io
from dotenv import load_dotenv

Extraindo o dataframe

In [364]:
df = pd.read_csv("../data/consultas_medicas.csv")

Realizando uma análise panorâmica sobre o df

In [365]:
df.sample(15)

,id_consulta,data_consulta,nome_paciente,nome_medico,especialidade,convenio,status,valor_consulta,avaliacao_paciente
85,86,2023-01-27,Isabela Cunha,Dra. Juliana Prado,Ortopedia,Unimed,Realizada,280.00,2.0
9,10,2023-05-08,Isabela Cunha,Dra. Renata Souza,Psiquiatria,Particular,Aguardando,300.00,NaN
119,120,2022-02-05,Henrique Pinto,Dr. Roberto Farias,Ortopedia,Unimed,Realizada,280.00,1.0
262,263,2023-08-10,CARLOS EDUARDO,Dra. Sofia Ramos,Neurologia,Particular,Realizada,400.00,1.0
7,8,24/09/2023,Fabio Correia,Marcos Vinicius,Dermatologia,SulAmérica,Realizada,NaN,3.0
83,84,2022-05-06,André Moreira,Dra. Renata Souza,Psiquiatria,amil,Realizada,300.00,2.0
52,53,2022-04-28,Henrique Pinto,Dra. Ana Beatriz,Cardiologia,SulAmérica,REALIZADA,350.00,5.0
108,109,2023-12-31,Ricardo Nunes,Dra. Sofia Ramos,Neurologia,SulAmérica,Aguardando,400.00,NaN
149,150,2023-09-14,Isabela Cunha,Dr. Roberto Farias,Ortopedia,SulAmérica,Cancelada,280.00,NaN
121,122,2022-07-10,Letícia Barros,Dra. Renata Souza,Psiquiatria,Particular,Aguardando,300.00,NaN


In [366]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_consulta         418 non-null    int64  
 1   data_consulta       400 non-null    object 
 2   nome_paciente       402 non-null    object 
 3   nome_medico         418 non-null    object 
 4   especialidade       418 non-null    object 
 5   convenio            403 non-null    object 
 6   status              418 non-null    object 
 7   valor_consulta      394 non-null    object 
 8   avaliacao_paciente  231 non-null    float64
dtypes: float64(1), int64(1), object(7)
memory usage: 29.5+ KB


In [367]:
df.describe()

,id_consulta,avaliacao_paciente
count,418.000000,231.000000
mean,209.500000,2.874459
std,120.810458,1.662056
min,1.000000,-1.000000
25%,105.250000,2.000000
50%,209.500000,3.000000
75%,313.750000,4.000000
max,418.000000,6.000000


In [368]:
int(df.duplicated(subset=df.columns.drop("id_consulta")).sum())

18

Limpeza do dataset

In [369]:
#Retirando linhas duplicadas
df = df.drop_duplicates(subset=df.columns.drop("id_consulta"))

In [370]:
df = df.dropna(subset=["data_consulta","nome_paciente", "convenio","status"])

In [371]:
df["avaliacao_paciente"] = df["avaliacao_paciente"].mask(df["avaliacao_paciente"] < 0 , "invalido")
df["avaliacao_paciente"] = df["avaliacao_paciente"].fillna("nao avaliado")

In [372]:
# Colocando essas colunas em minúsculo pois é mais fácil de trabalhar
cols = ["nome_paciente", "nome_medico", "especialidade","convenio", "status"]

df[cols] = df[cols].apply(lambda x: x.str.lower()).astype(str)

In [373]:
df["nome_medico"] = df["nome_medico"].str.replace(r"(?i)^dra?\.?\s*", "", regex=True)
df["valor_consulta"] = df["valor_consulta"].str.replace(r"(?i)^r\$?\s?","", regex=True).str.replace(",",".").astype(float)
df["data_consulta"] = pd.to_datetime(df["data_consulta"].replace("[/]","-", regex=True), dayfirst=True, format='mixed')

In [374]:
x = (
    df.groupby("nome_medico")["valor_consulta"]
    .mean()
    .reset_index()
)

x = x.set_index("nome_medico")["valor_consulta"].to_dict()

df["valor_consulta"] = df["valor_consulta"].fillna(df["nome_medico"].map(x))

In [375]:
# Adicionando uma ID UUIDv7
unic_ids = []

for i in range(len(df)):
    unic_ids.append(str(uuid6.uuid7()))

df["id_consulta"] = unic_ids

Carregamento do dataset na camada silver do PostgreSQL

In [376]:
buffer = io.StringIO()
df.to_csv(buffer, index=False)
buffer.seek(0)

0

In [377]:
load_dotenv(dotenv_path="../../.env")

conn = psycopg2.connect(
    host=os.getenv("host"),
    dbname=os.getenv("dbname"),
    user=os.getenv("user"),
    password=os.getenv("password"),
    port=os.getenv("port")
)

In [378]:
df.head(1)

,id_consulta,data_consulta,nome_paciente,nome_medico,especialidade,convenio,status,valor_consulta,avaliacao_paciente
0,019dbb3a-04f4-7c27-9125-6f5e2bf0a694,2023-08-19,aline monteiro,renata souza,psiquiatria,unimed,realizada,300.0,2.0


In [379]:
cur = conn.cursor()

cur.execute("CREATE SCHEMA IF NOT EXISTS case_2;")

cur.execute("""
CREATE TABLE IF NOT EXISTS case_2.consultas_silver (
    id_consulta VARCHAR(40) PRIMARY KEY,
    data_consulta TIMESTAMP,
    nome_paciente VARCHAR(255),
    nome_medico VARCHAR(255),
    especialidade VARCHAR(60),
    convenio VARCHAR(60),        
    status VARCHAR(30),
    valor_consulta DECIMAL(10, 2),
    avaliacao_paciente VARCHAR(20)
);
""")

cur.copy_expert(sql="COPY case_2.consultas_silver FROM STDIN WITH CSV HEADER", file=buffer)

conn.commit() 
cur.close() 
conn.close()

